# Notebook 4: Functional Evaluation

**Project:** Evaluating the Functional Correctness and Consistency of AI-Generated Introductory Programming Solutions: Implications for Computing Education

**Author:** Dr. C. V. Krishnaveni

**Institution:** Department of Computer Science, SKR & SKR Government College for Women (Autonomous), Kadapa, Andhra Pradesh, India

---

## Objective

This notebook evaluates the functional correctness of AI-generated Python programs using hidden test cases.

Each generated solution is executed automatically, and its output is compared against the expected output for every hidden test case.

The evaluation results are stored for subsequent error analysis.

## Workflow

This notebook performs the following tasks:

1. Clone the GitHub repository.
2. Load hidden test cases.
3. Locate all generated Python programs.
4. Execute each solution using hidden test cases.
5. Compare program outputs with expected outputs.
6. Record execution statistics.
7. Save evaluation results.

In [17]:
# ============================================================
# Clone Repository
# ============================================================

!rm -rf AI-Code_Judgement

!git clone https://github.com/vkchennuru/AI-Code_Judgement.git

Cloning into 'AI-Code_Judgement'...
remote: Enumerating objects: 157, done.
remote: Counting objects: 100% (157/157), done.
remote: Compressing objects: 100% (151/151), done.
remote: Total 157 (delta 59), reused 0 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (157/157), 389.51 KiB | 2.02 MiB/s, done.
Resolving deltas: 100% (59/59), done.


In [18]:
# ============================================================
# Import Libraries
# ============================================================

from pathlib import Path
import json
import subprocess
import tempfile
import pandas as pd
import time

In [19]:
# ============================================================
# Project Paths
# ============================================================

PROJECT_ROOT = Path("/content/AI-Code_Judgement")

DATASET_DIR = PROJECT_ROOT / "dataset"

CODE_DIR = PROJECT_ROOT / "generated_code"

RESULTS_DIR = PROJECT_ROOT / "results"

RESULTS_DIR.mkdir(parents=True, exist_ok=True)

print("Project Root :", PROJECT_ROOT)
print("Dataset      :", DATASET_DIR)
print("Generated Code:", CODE_DIR)
print("Results      :", RESULTS_DIR)

Project Root : /content/AI-Code_Judgement
Dataset      : /content/AI-Code_Judgement/dataset
Generated Code: /content/AI-Code_Judgement/generated_code
Results      : /content/AI-Code_Judgement/results


## Load Hidden Test Cases

In [20]:
with open(DATASET_DIR / "hidden_testcases.json", "r", encoding="utf-8") as f:
    hidden_tests = json.load(f)

print(f"Problems loaded : {len(hidden_tests)}")

Problems loaded : 10


## Locate Generated Programs

In [21]:
problem_folders = sorted(
    folder for folder in CODE_DIR.iterdir()
    if folder.is_dir()
)

print(f"Problems detected : {len(problem_folders)}\n")

for folder in problem_folders:
    print(folder.name)

Problems detected : 10

problem_001
problem_002
problem_003
problem_004
problem_005
problem_006
problem_007
problem_008
problem_009
problem_010


## Execute Python Programs

The following function executes an AI-generated Python program using a hidden test case and captures its output, execution status, runtime, and any runtime errors.

In [22]:
# ============================================================
# Execute a Python Program
# ============================================================

def execute_program(program_path, test_input, timeout=5):
    """
    Execute a Python program with the given input.

    Parameters
    ----------
    program_path : Path
        Path to the Python program.
    test_input : str
        Input supplied to the program.
    timeout : int
        Maximum execution time (seconds).

    Returns
    -------
    output : str
    success : bool
    runtime : float
    error : str
    """

    start_time = time.time()

    try:
        result = subprocess.run(
            ["python", str(program_path)],
            input=test_input,
            capture_output=True,
            text=True,
            timeout=timeout
        )

        runtime = time.time() - start_time

        if result.returncode == 0:
            return (
                result.stdout.strip(),
                True,
                runtime,
                ""
            )

        return (
            result.stderr.strip(),
            False,
            runtime,
            result.stderr.strip()
        )

    except subprocess.TimeoutExpired:
        runtime = time.time() - start_time
        return (
            "",
            False,
            runtime,
            "Timeout"
        )

    except Exception as e:
        runtime = time.time() - start_time
        return (
            "",
            False,
            runtime,
            str(e)
        )

## Verify Program Execution

Run one generated solution with a sample hidden test case to ensure that the execution function is working correctly before evaluating the complete benchmark.

In [23]:
# Select the first problem
sample_problem = problem_folders[0]

print(sample_problem)

/content/AI-Code_Judgement/generated_code/problem_001


## Evaluate AI-Generated Programs

Each generated Python program is executed against all hidden test cases for its corresponding programming problem.

For every solution, the notebook records:

- Execution success
- Number of passed test cases
- Total test cases
- Runtime
- Error type (if any)

The results are stored for subsequent statistical analysis.

In [24]:
# ============================================================
# Evaluate All Generated Programs
# ============================================================

evaluation_results = []

solution_files = ["solution_A.py", "solution_B.py", "solution_C.py"]

In [25]:
# ============================================================
# Evaluate All Generated Programs
# ============================================================

evaluation_results = []

solution_files = ["solution_A.py", "solution_B.py", "solution_C.py"]

for problem_folder in problem_folders:

    problem_id = int(problem_folder.name.split("_")[1])

    # Get all hidden test cases for this problem
    testcases = hidden_tests[str(problem_id)]["test_cases"]

    for solution in solution_files:

        program_path = problem_folder / solution

        if not program_path.exists():
            continue

        passed = 0
        total = len(testcases)

        execution_success = True
        error_type = ""

        total_runtime = 0

        for testcase in testcases:

            output, success, runtime, error = execute_program(
                program_path,
                testcase["input"]
            )

            total_runtime += runtime

            if success:

                if output.strip() == testcase["expected_output"].strip():
                    passed += 1

            else:

                execution_success = False
                error_type = error

        evaluation_results.append({

            "problem_id": problem_id,

            "solution": solution,

            "execution_success": execution_success,

            "passed_tests": passed,

            "total_tests": total,

            "accuracy": round(passed / total * 100, 2),

            "runtime_seconds": round(total_runtime, 4),

            "error_type": error_type

        })

In [26]:
# ============================================================
# Create Evaluation DataFrame
# ============================================================

evaluation_df = pd.DataFrame(evaluation_results)

print(f"Programs Evaluated : {len(evaluation_df)}")

display(evaluation_df.head())

Programs Evaluated : 30


,problem_id,solution,execution_success,passed_tests,total_tests,accuracy,runtime_seconds,error_type
0,1,solution_A.py,True,5,5,100.0,0.1780,
1,1,solution_B.py,True,5,5,100.0,0.1941,
2,1,solution_C.py,True,5,5,100.0,0.1927,
3,2,solution_A.py,True,5,5,100.0,0.1774,
4,2,solution_B.py,False,0,5,0.0,0.1801,"File ""/content/AI-Code_Judgement/generated_cod..."


In [27]:
# ============================================================
# Evaluation Summary
# ============================================================

print("=" * 70)
print("Evaluation Summary")
print("=" * 70)

print("Programs Evaluated :", len(evaluation_df))

print("Successful Runs    :", evaluation_df["execution_success"].sum())

print("Failed Runs        :", (~evaluation_df["execution_success"]).sum())

print("Average Accuracy   :",
      round(evaluation_df["accuracy"].mean(), 2), "%")

Evaluation Summary
Programs Evaluated : 30
Successful Runs    : 17
Failed Runs        : 13
Average Accuracy   : 56.67 %


In [28]:
# ============================================================
# Save Evaluation Results
# ============================================================

results_file = RESULTS_DIR / "evaluation_results.csv"

evaluation_df.to_csv(results_file, index=False)

print(f"Evaluation results saved to:\n{results_file}")

Evaluation results saved to:
/content/AI-Code_Judgement/results/evaluation_results.csv


In [29]:
# ============================================================
# Accuracy by Solution
# ============================================================

summary = (
    evaluation_df
    .groupby("solution")["accuracy"]
    .mean()
    .reset_index()
)

display(summary)

,solution,accuracy
0,solution_A.py,70.0
1,solution_B.py,50.0
2,solution_C.py,50.0


## Reproducibility

All generated Python programs were evaluated using the same hidden test cases.

The evaluation pipeline:

- Executes each generated solution.
- Compares program output with the expected output.
- Records execution success, accuracy, runtime, and error information.
- Stores the results in `results/evaluation_results.csv`.

This standardized evaluation ensures reproducibility and supports subsequent statistical analysis.

## Conclusion

In this notebook:

- Hidden test cases were loaded successfully.
- AI-generated Python programs were executed automatically.
- Functional correctness was evaluated using hidden test cases.
- Execution statistics were recorded.
- Evaluation results were saved for further analysis.

The generated dataset will be used in Notebook 5 for error analysis and visualization.

In [30]:
print("=" * 70)
print("Notebook 4 Completed Successfully")
print("=" * 70)

print(f"Programs Evaluated : {len(evaluation_df)}")
print(f"Average Accuracy   : {evaluation_df['accuracy'].mean():.2f}%")
print(f"Results File       : {results_file}")

print("\nReady for Notebook 5: Error Analysis")

Notebook 4 Completed Successfully
Programs Evaluated : 30
Average Accuracy   : 56.67%
Results File       : /content/AI-Code_Judgement/results/evaluation_results.csv

Ready for Notebook 5: Error Analysis
